# Weak-lensing galaxy shape catalogue validation

## Maps

Contents.
- Creat convergence maps

> **_NOTE:_** Before running this notebook, set kernel to `main_set.ipynb'

In [ ]:
from scipy.ndimage.filters import gaussian_filter

from lenspack.utils import bin2d
from lenspack.image.inversion import ks93

In [ ]:
from sp_validation.survey import *
from sp_validation.util import *
from sp_validation.basic import *
from sp_validation.plots import *
from sp_validation.cosmology import *

# Pixelise ellipticities

In [ ]:
# Compute number of pixels
Nx = int(size_x_deg / pixel_size_emap_amin * 60)
Ny = int(size_y_deg / pixel_size_emap_amin * 60)
print_stats(f'Numbers of elipticity pixels for KS93 = ({Nx}, {Ny})', stats_file, verbose=verbose)

In [ ]:
# Bin in 2D
g1_tmp, g2_tmp = bin2d(
    x,
    y,
    npix=(Nx, Ny), 
    v=(g_corr_mc_ngmix[0], g_corr_mc_ngmix[1]),
    extent=(min_x, max_x, min_y, max_y)
)

g_corr_mc_ngmix_map = np.array([g1_tmp, g2_tmp])

# Create convergence maps

In [ ]:
# Transform gamma -> kappa using the Kaiser-Squires (1993) algorithm
kappaE, kappaB = ks93(g1_sign * g_corr_mc_ngmix_map[0], g2_sign * g_corr_mc_ngmix_map[1])

In [ ]:
# Smooth with Gaussian filter
kappaE_sm = gaussian_filter(kappaE, smoothing_scale_pix)
kappaB_sm = gaussian_filter(kappaB, smoothing_scale_pix)

# Get known cluster positions

In [ ]:
# Get cluster information

sp_base = f'{os.environ["HOME"]}/sp_validation'
for sc in ['cosmology']:
    script = os.path.join(sp_base, 'sp_validation', sc)
    %run $script


cluster_cat_name = 'HFI_PCCS_SZ-union_R2.08.fits.gz'
vos_dir = 'vos:cfis/cosmostat/cosmology/external/Planck'

clusters = get_clusters(cluster_cat_name, vos_dir, data_dir, name, verbose=verbose)

print_stats(f"{len(clusters['ra'])} clusters found in {name} footprint", stats_file, verbose=verbose)

In [ ]:
# Project cluster positions
x_cluster, y_cluster =  radec2xy(ra_ngmix_mean, dec_ngmix_mean, clusters['ra'], clusters['dec'])
clusters['x'] = x_cluster
clusters['y'] = y_cluster

# Plot maps

## Convergence maps

In [ ]:
title = '$\kappa_{\\rm E}$'
out_path = f'{plot_dir}/kappa_E.png'

vlim = plot_map(kappaE_sm, ra_ngmix, dec_ngmix, title, out_path, clusters=clusters)

In [ ]:
title = '$\kappa_{\\rm B}$'
out_path = f'{plot_dir}/kappa_B.png'

plot_map(kappaB_sm, ra_ngmix, dec_ngmix, title, out_path, vlim=vlim, clusters=clusters)

## Stacked convergence maps

In [ ]:
for sc in ['cosmology']:
    script = os.path.join(sp_base, 'sp_validation', sc)
    %run $script


radius = 5

# Stack galaxies
res_stack_mm = stack_mm3(
    ra_ngmix,
    dec_ngmix,
    g_corr_mc_ngmix[0],
    g_corr_mc_ngmix[1],
    w_ngmix,
    clusters['ra'],
    clusters['dec'],
    clusters['z'],
    radius=radius, n_match=1000000
)

In [ ]:
# Plot stacked galaxy density, to check how uniform distribution is. Sometimes at the edges the number
# of galaxies drops visibly

plt.figure(figsize=(10, 10))
plt.hexbin(res_stack_mm[0], res_stack_mm[1], gridsize=100, cmap='gist_stern')
cbar = plt.colorbar()
cbar.set_label('Number count', rotation=270)
plt.title('Density plot')

In [ ]:
# Bin stacked ellipticities

npix = 2048
e1map_stack, e2map_stack = bin2d(
    res_stack_mm[0],
    res_stack_mm[1],
    v=(res_stack_mm[2], -res_stack_mm[3]),
    w=res_stack_mm[4], 
    npix=npix
)

In [ ]:
# transform to gamma -> kappa via the aisers & Squires (1993) algorithm
kappaE_stack, kappaB_stack = ks93(e1map_stack, e2map_stack)

# Smooth
kappaE_stack_sm = gaussian_filter(kappaE_stack, smoothing_scale_pix)
kappaB_stack_sm = gaussian_filter(kappaB_stack, smoothing_scale_pix)

In [ ]:
title = 'kappa_E'
output_path = f'{plot_dir}/kappaE_stacked.png'

vlim = plot_map_stacked(kappaE_stack_sm, title, output_path)

In [ ]:
title = 'kappa_B'
output_path = f'{plot_dir}/kappaB_stacked.png'

vlim = plot_map_stacked(kappaB_stack_sm, title, output_path, vlim=vlim)